# Traffic Analytics — Colab đầy đủ v3

Bộ ZIP đã gộp dataset, checkpoint, notebook dùng tài nguyên GPU và chế độ log gọn. Đặt **traffic_colab_complete_v3.zip** cùng thư mục với notebook này (không dùng Google Drive). Mở notebook này trên Colab, chọn runtime GPU rồi **Run all**.

Mặc định **batch 16**, CPU threads/workers theo số CPU Colab cấp; không áp trần RAM 10 GB hay CPU 50%. Giữ 534 ảnh, bốn lớp và cấu hình tích hợp 1 FPS. Tiếp tục YOLO11s rồi train YOLO26s, lưu kết quả trong thư mục notebook.

Màn hình chỉ hiện các bước chính, epoch đầu, mỗi 10 epoch và epoch cuối. Log đầy đủ được lưu ở **DS_Group13_training/runs/logs/** trong thư mục notebook; khi có lỗi sẽ hiện phần cuối log. Đổi `LOG_EVERY` nếu muốn xem thường xuyên hơn.

Nếu đang train bằng notebook cũ, đợi lưu checkpoint epoch mới rồi dừng ô training cũ trước khi dùng bản này. Batch thay đổi có thể ảnh hưởng kết quả; tiến độ và optimizer vẫn được khôi phục từ checkpoint.


In [ ]:
from pathlib import Path
import sys, os, json, hashlib, zipfile, subprocess
from datetime import datetime

# Dùng thư mục chứa notebook này thay cho Google Drive.
BASE_DIR = Path.cwd()
BUNDLE_PATH = BASE_DIR / 'traffic_colab_complete_v3.zip'
REPO = BASE_DIR / 'DS_Group13_colab_v3'
RUNS_DIR = BASE_DIR / 'DS_Group13_training' / 'runs'
RELEASE_ID = 'traffic-colab-complete-v3'
RUN_GROUP = 'handoff_01'  # Đổi tên khi bắt đầu một thí nghiệm mới.
RUN_TRAINING = True  # Người dùng đã chọn train; ô train vẫn kiểm tra GPU trước khi chạy.
START_YOLO11_FROM = 'checkpoint'  # Hoặc 'pretrained' cho thí nghiệm mới.
TRAIN_YOLO26 = True
EPOCHS = 100  # Tổng ngân sách, bao gồm các epoch đã train trước checkpoint.
BATCH = 16  # Có thể tăng 32 khi đã kiểm tra mức VRAM trong training.
LOG_EVERY = 10  # Chỉ hiện epoch đầu, mỗi 10 epoch và epoch cuối.
CPU_THREADS = os.cpu_count() or 1
WORKERS = CPU_THREADS
os.environ['PYTHONUNBUFFERED'] = '1'
for key in ('TRAIN_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS',
            'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[key] = str(CPU_THREADS)
print(f'Không áp trần CPU/RAM: batch={BATCH}, CPU threads={CPU_THREADS}, workers={WORKERS}')

Mounted at /content/drive
Không áp trần CPU/RAM: batch=16, CPU threads=2, workers=2


## Xác minh và giải nén

Kiểm tra SHA256 dữ liệu, code, notebook và checkpoint; dùng thư mục v3 riêng. Chạy lại bước chuẩn bị không ghi đè báo cáo nguồn.


In [ ]:
import shutil, stat

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def verified_archive_bytes(z, relative, expected):
    value = z.read('DS_Group13_colab/' + relative)
    if len(value) != expected['size'] or hashlib.sha256(value).hexdigest() != expected['sha256']:
        raise ValueError(f'Checksum trong ZIP không đúng: {relative}')
    return value

if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(f'Đặt ZIP vào thư mục notebook hoặc sửa BUNDLE_PATH: {BUNDLE_PATH}')
archive_sha = sha256_file(BUNDLE_PATH)
archive_marker = REPO / '.bundle_archive_sha256'
if archive_marker.is_file() and archive_marker.read_text().strip() != archive_sha:
    raise RuntimeError('Thư mục REPO thuộc ZIP khác. Chọn REPO mới để giữ nguyên dữ liệu hiện tại.')

# Đọc manifest gốc từ ZIP; audit.json và data.yaml trong thư mục làm việc có thể được tạo lại.
DATA_CONFIG = 'data/kaggle/prepared/vehicles/data.yaml'
RUNTIME_GENERATED = {'reports/data/audit.json', DATA_CONFIG}
with zipfile.ZipFile(BUNDLE_PATH) as z:
    prefix = 'DS_Group13_colab/'
    for member in z.infolist():
        path = Path(member.filename)
        if (path.is_absolute() or '..' in path.parts or '\\' in member.filename
                or ':' in member.filename or not member.filename.startswith(prefix)
                or stat.S_ISLNK(member.external_attr >> 16)):
            raise ValueError(f'Đường dẫn ZIP không hợp lệ: {member.filename}')
        target = (REPO / member.filename[len(prefix):]).resolve()
        if not target.is_relative_to(REPO.resolve()):
            raise ValueError(f'Đường dẫn ZIP vượt ra ngoài REPO: {member.filename}')
    manifest = json.loads(z.read(prefix + 'bundle_manifest.json'))
    if manifest.get('release_id') != RELEASE_ID:
        raise ValueError('Sai phiên bản ZIP. Dùng traffic_colab_complete_v3.zip cùng notebook v3 này.')
    notebook_path = 'notebooks/train_colab_complete_v3.ipynb'
    notebook_item = next((item for item in manifest['files'] if item['path'] == notebook_path), None)
    if (manifest.get('notebook_path') != notebook_path or notebook_item is None
            or notebook_item['sha256'] != manifest.get('notebook_sha256')):
        raise ValueError('Manifest không khớp notebook của bộ v3.')
    verified_archive_bytes(z, notebook_path, notebook_item)
    # Khôi phục file còn thiếu khi lần giải nén trước bị ngắt; giữ nguyên file đang có.
    for member in z.infolist():
        relative = member.filename[len(prefix):]
        if not relative or member.is_dir():
            continue
        target = REPO / relative
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(member) as source, target.open('xb') as destination:
                shutil.copyfileobj(source, destination, 1024 * 1024)
    seen = set()
    for item in manifest['files']:
        relative = item['path']
        target = (REPO / relative).resolve()
        if relative in seen or not target.is_relative_to(REPO.resolve()):
            raise ValueError(f'Đường dẫn manifest không hợp lệ hoặc bị lặp: {relative}')
        seen.add(relative)
        if relative in RUNTIME_GENERATED:
            baseline = verified_archive_bytes(z, relative, item)
            if relative == DATA_CONFIG:
                # Luôn lấy bản gốc đã xác minh; không tin backup từ lần chạy bị ngắt.
                (REPO / 'original_data.yaml').write_bytes(baseline)
            continue
        if not target.is_file() or target.stat().st_size != item['size'] or sha256_file(target) != item['sha256']:
            raise ValueError(f'Checksum không đúng: {relative}')
archive_marker.write_text(archive_sha)
print('Đã xác minh', len(manifest['files']), 'file nguồn; dữ liệu và checkpoint được kiểm tra đầy đủ.')

sys.path.insert(0, str(REPO / 'scripts'))
from colab_output import run_logged_command


Đã xác minh 1102 file nguồn; dữ liệu và checkpoint được kiểm tra đầy đủ.


## Thư viện và GPU

Cài các thư viện trong ZIP; giữ PyTorch/CUDA do Colab cung cấp. Chỉ chạy training khi CUDA thực sự khả dụng.


In [ ]:
run_logged_command(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-colab.txt')],
    cwd=REPO, log_path=REPO / 'reports/runtime/setup.log', label='Cài thư viện',
)
import torch, yaml
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | PyTorch: {torch.__version__}')
else:
    print('Chưa có CUDA: chọn Runtime → Change runtime type → GPU trước khi train.')


[Cài thư viện] started | full log: /content/DS_Group13_colab_v3/reports/runtime/setup.log
[Cài thư viện] completed in 5.2s | full log: /content/DS_Group13_colab_v3/reports/runtime/setup.log
GPU: Tesla T4 | PyTorch: 2.11.0+cu130


## Cập nhật đường dẫn Linux và audit dữ liệu

Ảnh, nhãn và cách chia tập được giữ nguyên. Báo cáo mới nằm tại `reports/runtime/audit.json`.


In [ ]:
data_dir = REPO / 'data/kaggle/prepared/vehicles'
DATA_YAML = data_dir / 'data.yaml'
# original_data.yaml luôn được ô kiểm tra tạo từ bản gốc đã xác minh trong ZIP.
config = yaml.safe_load((REPO / 'original_data.yaml').read_text())
config['path'] = str(data_dir)
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False))

# Không ghi đè audit.json đã được đóng gói và có checksum.
runtime_config = yaml.safe_load((REPO / 'configs/dataset.yaml').read_text())
runtime_config['report_dir'] = str(REPO / 'reports/runtime')
runtime_config_path = REPO / '.colab_runtime_dataset.yaml'
runtime_config_path.write_text(yaml.safe_dump(runtime_config, sort_keys=False))
run_logged_command(
    [sys.executable, str(REPO / 'scripts/audit_dataset.py'), '--config', str(runtime_config_path)],
    cwd=REPO, log_path=REPO / 'reports/runtime/audit.log', label='Audit dữ liệu',
)
runtime_audit = json.loads((REPO / 'reports/runtime/audit.json').read_text())
if not runtime_audit.get('passed'):
    raise RuntimeError('Dataset audit chưa đạt; xem reports/runtime/audit.json.')
print(f"Dữ liệu đạt: {runtime_audit['images_checked']} ảnh, train/val/test = 374/107/53.")


[Audit dữ liệu] started | full log: /content/DS_Group13_colab_v3/reports/runtime/audit.log
[Audit dữ liệu] completed in 10.8s | full log: /content/DS_Group13_colab_v3/reports/runtime/audit.log
Dữ liệu đạt: 534 ảnh, train/val/test = 374/107/53.


## Nguồn checkpoint và kiểm tra trước khi train

Ưu tiên `last.pt` trong `RUNS_DIR` của đúng `RUN_GROUP`; nếu chưa có, dùng snapshot trong ZIP cho YOLO11s. Snapshot đã hoàn thành 13 epoch, nên tiếp tục ở epoch 14; checkpoint trong `RUNS_DIR` mới hơn sẽ tiếp tục từ tiến độ của nó. YOLO26s bắt đầu từ pretrained nếu chưa có checkpoint trong `RUNS_DIR`.

Giữ `RUN_GROUP='handoff_01'` để tiếp tục lượt trước. Chỉ đổi nhóm khi chủ động bắt đầu thí nghiệm mới. Lượt đã có trạng thái hoàn tất được bỏ qua. Các lệnh dưới đây mới kiểm tra dữ liệu/cấu hình.


In [ ]:
if START_YOLO11_FROM not in {'checkpoint', 'pretrained'}:
    raise ValueError('Chọn checkpoint hoặc pretrained')
JOB_NAMES = {'yolo11s': f'{RUN_GROUP}_yolo11s', 'yolo26s': f'{RUN_GROUP}_yolo26s'}
SNAPSHOT = REPO / manifest['source_run'] / 'weights/last.pt'

def make_command(model, start_mode):
    run_name = JOB_NAMES[model]
    saved_last = RUNS_DIR / run_name / 'weights/last.pt'
    command = [sys.executable, str(REPO / 'scripts/train_colab.py'),
               '--model', model, '--data', str(DATA_YAML),
               '--project', str(RUNS_DIR), '--name', run_name,
               '--epochs', str(EPOCHS), '--batch', str(BATCH), '--workers', str(WORKERS), '--device', '0',
               '--staging-dir', str(REPO.parent / 'traffic_colab_staging_v3'), '--compact-progress']
    if saved_last.is_file():
        command += ['--resume', str(saved_last)]
    elif model == 'yolo11s' and start_mode == 'checkpoint':
        command += ['--resume', str(SNAPSHOT)]
    else:
        command += ['--weights', str(REPO / 'weights' / f'{model}.pt')]
    return command

jobs = [('yolo11s', START_YOLO11_FROM)]
if TRAIN_YOLO26:
    jobs.append(('yolo26s', 'pretrained'))
pending = []
for model, mode in jobs:
    marker = RUNS_DIR / JOB_NAMES[model] / 'colab_run_status.json'
    completed = json.loads(marker.read_text()).get('status') == 'completed' if marker.is_file() else False
    if completed:
        print('Đã hoàn tất, bỏ qua:', model)
    else:
        pending.append((model, mode))
jobs = pending
for model, mode in jobs:
    run_logged_command(
        make_command(model, mode), cwd=REPO,
        log_path=REPO / 'reports/runtime' / f'preflight_{model}.log',
        label=f'Kiểm tra {model}',
    )


[Kiểm tra yolo11s] started | full log: /content/DS_Group13_colab_v3/reports/runtime/preflight_yolo11s.log
[Kiểm tra yolo11s] completed in 0.2s | full log: /content/DS_Group13_colab_v3/reports/runtime/preflight_yolo11s.log
[Kiểm tra yolo26s] started | full log: /content/DS_Group13_colab_v3/reports/runtime/preflight_yolo26s.log
[Kiểm tra yolo26s] completed in 0.2s | full log: /content/DS_Group13_colab_v3/reports/runtime/preflight_yolo26s.log


## Training GPU — log gọn

`RUN_TRAINING=True` đã bật. Hai model chạy tuần tự; ngân sách tổng 100 epoch, early stopping có thể kết thúc sớm. Mặc định chỉ hiện epoch đầu, mỗi 10 epoch và epoch cuối. Log đầy đủ được ghi nối tiếp trong `RUNS_DIR/logs`, tách khỏi thư mục checkpoint của từng model.

Helper giữ các phần sửa checksum/optimizer, `channels_last=False`, epoch, EMA và best checkpoint. Không cần file vá rời. Không khởi chạy train local.


In [ ]:
if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError('Không có CUDA GPU; chọn GPU runtime trước khi train.')
    if not jobs:
        print('Các model đã hoàn thành; xem checkpoint trong RUNS_DIR.')
    for model, mode in jobs:
        run_logged_command(
            [*make_command(model, mode), '--execute'], cwd=REPO,
            log_path=RUNS_DIR / 'logs' / f'{JOB_NAMES[model]}.log',
            label=model, summarize=True, progress_every=LOG_EVERY,
        )
else:
    print('Chưa train: đặt RUN_TRAINING=True rồi chạy lại ô này.')


[yolo11s] started | full log: /content/drive/MyDrive/DS_Group13_training/runs/logs/handoff_01_yolo11s.log
[yolo11s] epoch 14/100 | mAP50=0.2787 | mAP50-95=0.1720 | P=0.5694 | R=0.2488
[yolo11s] epoch 20/100 | mAP50=0.3006 | mAP50-95=0.1630 | P=0.4807 | R=0.3252
[yolo11s] epoch 30/100 | mAP50=0.4262 | mAP50-95=0.2659 | P=0.4167 | R=0.5095
[yolo11s] epoch 40/100 | mAP50=0.5249 | mAP50-95=0.3337 | P=0.5361 | R=0.5030
[yolo11s] epoch 50/100 | mAP50=0.5026 | mAP50-95=0.3377 | P=0.6320 | R=0.4854
[yolo11s] epoch 60/100 | mAP50=0.5167 | mAP50-95=0.3557 | P=0.5929 | R=0.4737
[yolo11s] epoch 70/100 | mAP50=0.5307 | mAP50-95=0.3565 | P=0.6386 | R=0.5264
[yolo11s] epoch 80/100 | mAP50=0.5217 | mAP50-95=0.3707 | P=0.6692 | R=0.4739
[yolo11s] epoch 90/100 | mAP50=0.5455 | mAP50-95=0.3672 | P=0.7004 | R=0.5137
[yolo11s] epoch 91/100 | mAP50=0.5304 | mAP50-95=0.3569 | P=0.6962 | R=0.5080
[yolo11s] completed in 1502.7s | full log: /content/drive/MyDrive/DS_Group13_training/runs/logs/handoff_01_yolo11s

## Kiểm tra kết quả

Dùng validation để chọn model; chỉ đánh giá test sau khi chốt checkpoint. Khi chạy lại trên runtime mới, notebook ưu tiên checkpoint trong `RUNS_DIR` và bỏ qua model đã hoàn tất.


In [ ]:
import csv
for model in JOB_NAMES:
    run = RUNS_DIR / JOB_NAMES[model]
    result = run / 'results.csv'
    summary = 'chưa có metrics'
    if result.is_file():
        with result.open(newline='') as f:
            rows = list(csv.DictReader(f))
        if rows:
            latest = {key.strip(): value for key, value in rows[-1].items()}
            summary = f"epoch={latest.get('epoch', '?')}, mAP50={latest.get('metrics/mAP50(B)', '?')}"
    saved = ', '.join(name for name in ('best.pt', 'last.pt') if (run / 'weights' / name).is_file())
    print(f"{model}: {summary}; checkpoint: {saved or 'chưa có'}")
print('Kết quả:', RUNS_DIR)
print('Log đầy đủ:', RUNS_DIR / 'logs')


yolo11s: epoch=91, mAP50=0.53043; checkpoint: best.pt, last.pt
yolo26s: epoch=21, mAP50=0.47036; checkpoint: best.pt, last.pt
Kết quả: /content/drive/MyDrive/DS_Group13_training/runs
Log đầy đủ: /content/drive/MyDrive/DS_Group13_training/runs/logs
